![image_1781181529065.png](./image_1781181529065.png "image_1781181529065.png")

![image_1781181547685.png](./image_1781181547685.png "image_1781181547685.png")

![image_1781181561827.png](./image_1781181561827.png "image_1781181561827.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import Window
# Initialize Spark session
spark = SparkSession.builder.appName("ProductsDataFrame").getOrCreate()

# Define the dataset
data = [
    (1, "Laptop Pro", "Electronics", 50000),
    (2, "Wireless Mouse", "Electronics", 15000),
    (3, "USB-C Hub", "Electronics", 10000),
    (4, "Keyboard", "Electronics", 25000),
    (5, "Running Shoes", "Apparel", 30000),
    (6, "Winter Jacket", "Apparel", 45000),
    (7, "Sports T-Shirt", "Apparel", 15000),
    (8, "Yoga Pants", "Apparel", 10000)
]

# Define schema
columns = ["id", "name", "category", "revenue"]

# Create DataFrame
products_df = spark.createDataFrame(data, columns)

# Show DataFrame
products_df.show()


In [0]:
result_df = (
    products_df.withColumn(
        "category_sum", f.sum("revenue").over(Window.partitionBy("category"))
    )
    .withColumn(
        "running_total",
        f.sum("revenue").over(
            Window.partitionBy("category")
            .orderBy(f.desc("revenue"))
            .rowsBetween(Window.unboundedPreceding, Window.currentRow)
        ),
    )
    .withColumn(
        "cumulative_share_pct",
        f.round(
            f.col("running_total")
            * 100
            / f.sum("revenue").over(Window.partitionBy("category")),
            1,
        ),
    )
    .select(
        f.col("category"),
        f.col("name"),
        f.col("revenue"),
        f.col("cumulative_share_pct"),
    )
)
display(result_df)